# TactiQ — Phase 2 Feature Exploration
Sanity checks and visual exploration of `team_style_profiles`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import psycopg2
import seaborn as sns
from dotenv import load_dotenv

load_dotenv(Path('../.env'))

conn = psycopg2.connect(
    host=os.getenv('DB_HOST','localhost'),
    port=int(os.getenv('DB_PORT',5432)),
    dbname=os.getenv('DB_NAME','tactiq'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
)

df = pd.read_sql('SELECT * FROM team_style_profiles', conn)
print(f'Teams loaded: {len(df)}')
df.head()

In [ ]:
# Cell 2 — Correlation heatmap
numeric_cols = [
    'avg_possession_pct','avg_pass_completion_pct','avg_progressive_passes_p90',
    'avg_passes_final_third_p90','avg_ppda','avg_pressure_success_rate',
    'avg_total_pressures_p90','avg_xg_created_p90','avg_xg_conceded_p90',
    'avg_xg_ratio','avg_progressive_carry_pct','avg_carries_final_third_p90',
    'avg_pass_completion_under_pressure_pct','avg_set_piece_shot_pct','win_rate'
]

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    df[numeric_cols].corr(),
    annot=True, fmt='.2f', cmap='RdYlGn', center=0,
    linewidths=0.5, ax=ax
)
ax.set_title('Feature Correlation Heatmap — Team Style Profiles', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 3 — Distribution histograms for core features
core_features = [
    ('avg_possession_pct',       'Avg Possession %'),
    ('avg_pass_completion_pct',  'Avg Pass Completion %'),
    ('avg_ppda',                 'Avg PPDA'),
    ('avg_xg_created_p90',       'Avg xG Created'),
    ('avg_xg_conceded_p90',      'Avg xG Conceded'),
    ('avg_xg_ratio',             'Avg xG Ratio'),
    ('avg_progressive_carry_pct','Avg Progressive Carry %'),
    ('avg_total_pressures_p90',  'Avg Total Pressures'),
    ('avg_passes_final_third_p90','Avg Passes Into Final Third'),
    ('win_rate',                 'Win Rate'),
]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, (col, label) in zip(axes.flat, core_features):
    data = df[col].dropna()
    ax.hist(data, bins=15, edgecolor='white', color='steelblue', alpha=0.85)
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('')
plt.suptitle('Feature Distributions — Team Style Profiles', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4 — Top/Bottom 5 teams per metric
ranking_metrics = [
    ('avg_ppda',                 'PPDA',                  'asc'),
    ('avg_xg_ratio',             'xG Ratio',              'desc'),
    ('avg_possession_pct',       'Possession %',          'desc'),
    ('avg_xg_created_p90',       'xG Created',            'desc'),
    ('avg_progressive_carry_pct','Progressive Carry %',   'desc'),
]

for col, label, order in ranking_metrics:
    asc = (order == 'asc')
    top5 = df[['team_name', col]].dropna().sort_values(col, ascending=asc).head(5)
    print(f'\n--- Top 5 by {label} ({"lower=better" if asc else "higher=better"}) ---')
    for _, row in top5.iterrows():
        print(f'  {row["team_name"]:<25} {row[col]:.3f}')

In [ ]:
# Cell 5 — Scatter: PPDA vs Possession
fig, ax = plt.subplots(figsize=(10, 7))
plot_df = df[['team_name','avg_ppda','avg_possession_pct']].dropna()

ax.scatter(plot_df['avg_ppda'], plot_df['avg_possession_pct'],
           s=80, alpha=0.75, color='steelblue', edgecolors='white', linewidths=0.6)

for _, row in plot_df.iterrows():
    ax.annotate(row['team_name'], (row['avg_ppda'], row['avg_possession_pct']),
                fontsize=7, alpha=0.85, xytext=(3, 3), textcoords='offset points')

ax.axvline(plot_df['avg_ppda'].median(), color='gray', linestyle='--', alpha=0.5, label='PPDA median')
ax.axhline(50, color='gray', linestyle=':', alpha=0.5, label='50% possession')
ax.set_xlabel('Avg PPDA (lower = more pressing)', fontsize=11)
ax.set_ylabel('Avg Possession %', fontsize=11)
ax.set_title('PPDA vs Possession — Do High-Press Teams Dominate Possession?', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Scatter: xG Created vs xG Conceded (quadrant plot)
fig, ax = plt.subplots(figsize=(10, 7))
plot_df = df[['team_name','avg_xg_created_p90','avg_xg_conceded_p90']].dropna()

ax.scatter(plot_df['avg_xg_conceded_p90'], plot_df['avg_xg_created_p90'],
           s=80, alpha=0.75, color='steelblue', edgecolors='white', linewidths=0.6)

for _, row in plot_df.iterrows():
    ax.annotate(row['team_name'],
                (row['avg_xg_conceded_p90'], row['avg_xg_created_p90']),
                fontsize=7, alpha=0.85, xytext=(3, 3), textcoords='offset points')

med_x = plot_df['avg_xg_conceded_p90'].median()
med_y = plot_df['avg_xg_created_p90'].median()
ax.axvline(med_x, color='gray', linestyle='--', alpha=0.5)
ax.axhline(med_y, color='gray', linestyle='--', alpha=0.5)

ax.text(ax.get_xlim()[0]*1.02, med_y*1.05, 'Elite\n(create+, concede-)', fontsize=8, color='green', alpha=0.7)
ax.text(med_x*1.02, ax.get_ylim()[0]*1.02, 'Vulnerable\n(create-, concede+)', fontsize=8, color='red', alpha=0.7)

ax.set_xlabel('Avg xG Conceded', fontsize=11)
ax.set_ylabel('Avg xG Created', fontsize=11)
ax.set_title('xG Created vs xG Conceded — Tactical Quadrant Plot', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Feature range summary
summary = df[numeric_cols].describe().T
summary.columns = ['count','mean','std','min','25%','50%','75%','max']
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 120)
print('\n=== Feature Range Summary ===')
print(summary.to_string())